# Project 3: Retail Demand Forecasting (Walmart M5)

This notebook is designed to run directly on Kaggle using the official **M5 Forecasting - Accuracy** dataset.

## Objective
Instead of applying a single algorithm to all products, we simulate a real-world supply chain where products have different demand profiles. We will cluster our time-series into three categories:
1. **Steady Demand** -> Forecasted with **ARIMA**
2. **Seasonal Demand** -> Forecasted with **Facebook Prophet**
3. **Volatile/Intermittent Demand** -> Forecasted with a Deep Learning **PyTorch LSTM**

This notebook demonstrates the "No Free Lunch" theorem by proving that classical stats, curve fitting, and deep learning each excel at different types of demand.

In [ ]:
!pip install prophet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

### 1. Load the Data
We load a small subset of the massive M5 dataset to keep execution fast. We aggregate the daily sales data for 3 specific representative items.

In [ ]:
print("Loading M5 Dataset...")
df = pd.read_csv('/kaggle/input/m5-forecasting-accuracy/sales_train_validation.csv')

# Select 3 items from California Store 1 to represent our 3 clusters
# 1. Steady Demand (e.g. Household essentials)
# 2. Seasonal Demand (e.g. Hobbies/Decor)
# 3. Volatile/Intermittent Demand (e.g. Niche items)

steady_item_id = 'FOODS_3_090_CA_1_validation'
seasonal_item_id = 'HOBBIES_1_234_CA_1_validation'
volatile_item_id = 'HOUSEHOLD_1_118_CA_1_validation'

steady_ts = df[df['id'] == steady_item_id].iloc[:, 6:].T
seasonal_ts = df[df['id'] == seasonal_item_id].iloc[:, 6:].T
volatile_ts = df[df['id'] == volatile_item_id].iloc[:, 6:].T

# M5 data spans 1913 days
dates = pd.date_range(start='2011-01-29', periods=1913, freq='D')
steady_ts.index = dates
seasonal_ts.index = dates
volatile_ts.index = dates

steady_ts.columns = ['Sales']
seasonal_ts.columns = ['Sales']
volatile_ts.columns = ['Sales']

fig, axes = plt.subplots(3, 1, figsize=(15, 10))
axes[0].plot(steady_ts[-365:], color='blue', label='Steady Demand')
axes[0].legend()
axes[1].plot(seasonal_ts[-365:], color='green', label='Seasonal Demand')
axes[1].legend()
axes[2].plot(volatile_ts[-365:], color='red', label='Volatile Demand')
axes[2].legend()
plt.tight_layout()
plt.show()

### 2. ARIMA for Steady Demand
ARIMA is excellent for data with a clear trend and moving average, but struggles with extreme non-linearity.

In [ ]:
train_steady = steady_ts[:-30]
test_steady = steady_ts[-30:]

print("Training ARIMA...")
arima_model = ARIMA(train_steady, order=(5, 1, 0))
arima_fit = arima_model.fit()

arima_preds = arima_fit.forecast(steps=30)

plt.figure(figsize=(10,4))
plt.plot(test_steady.index, test_steady['Sales'], label='Actual')
plt.plot(test_steady.index, arima_preds, color='red', label='ARIMA Forecast')
plt.title('ARIMA on Steady Demand (30-day forecast)')
plt.legend()
plt.show()

### 3. Facebook Prophet for Seasonal Demand
Prophet was built specifically for business time-series that exhibit strong daily/weekly/yearly seasonality.

In [ ]:
prophet_df = seasonal_ts.reset_index()
prophet_df.columns = ['ds', 'y']
train_seasonal = prophet_df[:-30]
test_seasonal = prophet_df[-30:]

print("Training Prophet...")
m = Prophet(yearly_seasonality=True, weekly_seasonality=True)
m.fit(train_seasonal)

future = m.make_future_dataframe(periods=30)
forecast = m.predict(future)

plt.figure(figsize=(10,4))
plt.plot(test_seasonal['ds'], test_seasonal['y'], label='Actual')
plt.plot(test_seasonal['ds'], forecast['yhat'][-30:], color='green', label='Prophet Forecast')
plt.title('Prophet on Seasonal Demand (30-day forecast)')
plt.legend()
plt.show()

### 4. PyTorch LSTM for Volatile Demand
Neural Networks (like LSTMs) excel at identifying complex, non-linear relationships in highly volatile sequences where classical stats fail.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

scaler = MinMaxScaler(feature_range=(-1, 1))
scaled_data = scaler.fit_transform(volatile_ts.values)

def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data)-seq_length-1):
        xs.append(data[i:(i+seq_length)])
        ys.append(data[i+seq_length])
    return np.array(xs), np.array(ys)

seq_length = 30
X, y = create_sequences(scaled_data, seq_length)

train_size = len(X) - 30
X_train = torch.FloatTensor(X[:train_size]).to(device)
y_train = torch.FloatTensor(y[:train_size]).to(device)
X_test = torch.FloatTensor(X[train_size:]).to(device)
y_test = torch.FloatTensor(y[train_size:]).to(device)

class DemandLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, num_layers=2, batch_first=True)
        self.linear = nn.Linear(64, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.linear(out[:, -1, :])
        return out

model = DemandLSTM().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Training PyTorch LSTM...")
for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    out = model(X_train)
    loss = criterion(out, y_train)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    preds = model(X_test).cpu().numpy()

preds_inverse = scaler.inverse_transform(preds)
actual_inverse = scaler.inverse_transform(y_test.cpu().numpy())

plt.figure(figsize=(10,4))
plt.plot(volatile_ts.index[-30:], actual_inverse, label='Actual')
plt.plot(volatile_ts.index[-30:], preds_inverse, color='orange', label='LSTM Forecast')
plt.title('PyTorch LSTM on Volatile Demand (30-day forecast)')
plt.legend()
plt.show()